In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VOCAB = ['A', 'B', 'C', 'D', '|', '<SOS>', '<EOS>', '<PAD>']
stoi = {ch: i for i, ch in enumerate(VOCAB)}
itos = {i: ch for ch, i in stoi.items()}

VOCAB_SIZE = len(VOCAB)
MAX_LEN = 6  # максимум длины слова

EMB_SIZE = 32
NHEAD = 4
HID_DIM = 64
NLAYERS = 2

In [3]:
def generate_word():
    length = random.randint(1, MAX_LEN)
    return ''.join(random.choice('ABCD') for _ in range(length))

def encode(s):
    return [stoi[c] for c in s]

def decode(x):
    return ''.join(itos[i] for i in x if i not in [stoi['<EOS>'], stoi['<PAD>']])

def make_sample():
    x = generate_word()
    y = x + "|" + x
    return x, y

In [4]:
def pad(seq, max_len):
    seq = seq + [stoi['<EOS>']]
    seq += [stoi['<PAD>']] * (max_len - len(seq))
    return seq

def get_batch(batch_size=32):
    xs, ys = [], []

    for _ in range(batch_size):
        x, y = make_sample()

        xs.append(encode(x))
        ys.append([stoi['<SOS>']] + encode(y))

    max_x = max(len(x) + 1 for x in xs)
    max_y = max(len(y) + 1 for y in ys)

    xs = [pad(x, max_x) for x in xs]
    ys = [pad(y, max_y) for y in ys]

    return (
        torch.tensor(xs).T.to(DEVICE),
        torch.tensor(ys).T.to(DEVICE)
    )

In [5]:
class TransformerCopy(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(VOCAB_SIZE, EMB_SIZE)

        self.transformer = nn.Transformer(
            d_model=EMB_SIZE,
            nhead=NHEAD,
            num_encoder_layers=NLAYERS,
            num_decoder_layers=NLAYERS,
            dim_feedforward=HID_DIM
        )

        self.fc = nn.Linear(EMB_SIZE, VOCAB_SIZE)

    def forward(self, src, tgt):
        src = self.embedding(src)
        tgt = self.embedding(tgt)

        out = self.transformer(src, tgt)
        return self.fc(out)

model = TransformerCopy().to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=stoi['<PAD>'])
optimizer = optim.Adam(model.parameters(), lr=0.001)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


In [6]:
for epoch in range(1000):
    src, tgt = get_batch()

    optimizer.zero_grad()

    output = model(src, tgt[:-1])

    loss = criterion(
        output.reshape(-1, VOCAB_SIZE),
        tgt[1:].reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 2.2632
Epoch 100, Loss: 1.3136
Epoch 200, Loss: 1.2491
Epoch 300, Loss: 1.1848
Epoch 400, Loss: 1.2540
Epoch 500, Loss: 1.1991
Epoch 600, Loss: 1.1562
Epoch 700, Loss: 1.1602
Epoch 800, Loss: 1.1901
Epoch 900, Loss: 1.1694


In [7]:
def predict(word):
    src = torch.tensor([pad(encode(word), len(word)+1)]).T.to(DEVICE)

    tgt = torch.tensor([[stoi['<SOS>']]]).to(DEVICE)

    for _ in range(len(word)*2 + 2):
        out = model(src, tgt)
        next_token = out[-1].argmax(dim=-1).item()

        tgt = torch.cat([
            tgt,
            torch.tensor([[next_token]]).to(DEVICE)
        ], dim=0)

        if next_token == stoi['<EOS>']:
            break

    return decode(tgt[1:].squeeze().tolist())

In [8]:
test = "ABCD"
print(f"Вход: {test}")
print(f"Выход: {predict(test)}")

Вход: ABCD
Выход: BBBBBBBBBB


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import random

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ======================
# Словарь
# ======================
VOCAB = ['A', 'B', 'C', 'D', '|', '<SOS>', '<EOS>', '<PAD>']
stoi = {c: i for i, c in enumerate(VOCAB)}
itos = {i: c for c, i in stoi.items()}
VOCAB_SIZE = len(VOCAB)

MAX_LEN = 6
EMB = 64

In [ ]:
# ======================
# Positional Encoding
# ======================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)

        div = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.pe = pe.unsqueeze(1)

    def forward(self, x):
        return x + self.pe[:x.size(0)].to(x.device)

In [ ]:
# ======================
# Генерация данных
# ======================
def generate_word():
    n = random.randint(1, MAX_LEN)
    return ''.join(random.choice("ABCD") for _ in range(n))

def encode(s):
    return [stoi[c] for c in s]

def decode(seq):
    out = []
    for x in seq:
        if x == stoi['<EOS>']:
            break
        if x != stoi['<PAD>']:
            out.append(itos[x])
    return ''.join(out)

def pad(seq, L):
    return seq + [stoi['<PAD>']] * (L - len(seq))

def get_batch(batch_size=64):
    srcs, tgts = [], []

    for _ in range(batch_size):
        x = generate_word()
        y = x + "|" + x

        src = encode(x) + [stoi['<EOS>']]
        tgt = [stoi['<SOS>']] + encode(y) + [stoi['<EOS>']]

        srcs.append(src)
        tgts.append(tgt)

    max_src = max(len(s) for s in srcs)
    max_tgt = max(len(t) for t in tgts)

    srcs = [pad(s, max_src) for s in srcs]
    tgts = [pad(t, max_tgt) for t in tgts]

    return (
        torch.tensor(srcs).T.to(DEVICE),
        torch.tensor(tgts).T.to(DEVICE)
    )

In [ ]:
# ======================
# Модель
# ======================
class CopyTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.emb = nn.Embedding(VOCAB_SIZE, EMB)
        self.pos = PositionalEncoding(EMB)

        self.tr = nn.Transformer(
            d_model=EMB,
            nhead=4,
            num_encoder_layers=2,
            num_decoder_layers=2,
            dim_feedforward=128
        )

        self.fc = nn.Linear(EMB, VOCAB_SIZE)

    def forward(self, src, tgt):
        src = self.pos(self.emb(src))
        tgt = self.pos(self.emb(tgt))

        tgt_mask = self.tr.generate_square_subsequent_mask(
            tgt.size(0)
        ).to(DEVICE)

        out = self.tr(src, tgt, tgt_mask=tgt_mask)
        return self.fc(out)

model = CopyTransformer().to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=stoi['<PAD>'])
optimizer = optim.Adam(model.parameters(), lr=1e-3)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


In [ ]:
# ======================
# Обучение
# ======================
for epoch in range(2000):
    src, tgt = get_batch()

    optimizer.zero_grad()

    out = model(src, tgt[:-1])

    loss = criterion(
        out.reshape(-1, VOCAB_SIZE),
        tgt[1:].reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(epoch, loss.item())

0 2.103461265563965
200 0.1538483053445816
400 0.0218669380992651
600 0.03994888812303543
800 0.017236463725566864
1000 0.009778786450624466
1200 0.02597814053297043
1400 0.024745792150497437
1600 0.037907443940639496
1800 0.00506043154746294


In [ ]:
# ======================
# Предсказание
# ======================
def predict(word):
    src = torch.tensor(
        [encode(word) + [stoi['<EOS>']]]
    ).T.to(DEVICE)

    tgt = torch.tensor([[stoi['<SOS>']]]).to(DEVICE)

    for _ in range(len(word)*2 + 3):
        out = model(src, tgt)
        next_tok = out[-1].argmax(-1).item()

        tgt = torch.cat([
            tgt,
            torch.tensor([[next_tok]]).to(DEVICE)
        ])

        if next_tok == stoi['<EOS>']:
            break

    return decode(tgt[1:].squeeze().tolist())


In [ ]:
# Проверка
print(predict("ABCD"))
print(predict("BAC"))
print(predict("DDAB"))

ABCD|ABCD
BAC|BAC
DDAB|DDABB


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import random

DEVICE = "cpu"

# ==========================
# Параметры модели
# ==========================
VOCAB = ['A', 'B', 'C', 'D', '|', '<SOS>', '<EOS>', '<PAD>']
stoi = {c: i for i, c in enumerate(VOCAB)}
itos = {i: c for c, i in stoi.items()}

VOCAB_SIZE = len(VOCAB)
D_MODEL = 6
N_HEADS = 2
N_LAYERS = 2
SEQ_LEN = 10

# ==========================
# Positional Encoding
# ==========================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=SEQ_LEN):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** (i / d_model)))
                if i + 1 < d_model:
                    pe[pos, i + 1] = math.cos(pos / (10000 ** (i / d_model)))

        self.pe = pe.unsqueeze(1)

    def forward(self, x):
        return x + self.pe[:x.size(0)].to(x.device)

# ==========================
# Генерация данных
# ==========================
MAX_INPUT_LEN = 3

def generate_word():
    n = random.randint(1, MAX_INPUT_LEN)
    return ''.join(random.choice("ABCD") for _ in range(n))

def encode(s):
    return [stoi[c] for c in s]

def decode(seq):
    out = []
    for x in seq:
        if x == stoi['<EOS>']:
            break
        if x not in [stoi['<PAD>'], stoi['<SOS>']]:
            out.append(itos[x])
    return ''.join(out)

def pad(seq):
    if len(seq) > SEQ_LEN:
        seq = seq[:SEQ_LEN]
    return seq + [stoi['<PAD>']] * (SEQ_LEN - len(seq))

def get_batch(batch_size=32):
    srcs, tgts = [], []

    for _ in range(batch_size):
        x = generate_word()
        y = x + "|" + x

        src = encode(x) + [stoi['<EOS>']]
        tgt = [stoi['<SOS>']] + encode(y) + [stoi['<EOS>']]

        srcs.append(pad(src))
        tgts.append(pad(tgt))

    return (
        torch.tensor(srcs, dtype=torch.long).T,
        torch.tensor(tgts, dtype=torch.long).T
    )

# ==========================
# Transformer
# ==========================
class TinyTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.embed = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.pos = PositionalEncoding(D_MODEL)

        self.tr = nn.Transformer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            num_encoder_layers=N_LAYERS,
            num_decoder_layers=N_LAYERS,
            dim_feedforward=24
        )

        self.fc = nn.Linear(D_MODEL, VOCAB_SIZE)

    def forward(self, src, tgt):
        src = self.pos(self.embed(src))
        tgt = self.pos(self.embed(tgt))

        mask = self.tr.generate_square_subsequent_mask(tgt.size(0))

        out = self.tr(src, tgt, tgt_mask=mask)
        return self.fc(out)

model = TinyTransformer().to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=stoi['<PAD>'])
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ==========================
# Обучение
# ==========================
for epoch in range(3000):
    src, tgt = get_batch()

    optimizer.zero_grad()

    out = model(src, tgt[:-1])

    loss = criterion(
        out.reshape(-1, VOCAB_SIZE),
        tgt[1:].reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if epoch % 300 == 0:
        print(epoch, loss.item())

# ==========================
# Предсказание
# ==========================
def predict(word):
    src = pad(encode(word) + [stoi['<EOS>']])
    src = torch.tensor([src]).T

    tgt = torch.tensor([[stoi['<SOS>']]])

    for _ in range(SEQ_LEN):
        out = model(src, tgt)
        next_token = out[-1].argmax(-1).item()

        tgt = torch.cat([
            tgt,
            torch.tensor([[next_token]])
        ])

        if next_token == stoi['<EOS>']:
            break

    return decode(tgt.squeeze().tolist())

# ==========================
# Тест
# ==========================
print(predict("AB"))
print(predict("BCD"))
print(predict("DACA"))

0 2.1701741218566895
300 0.40450093150138855
600 0.14308442175388336
900 0.09903490543365479
1200 0.030457884073257446
1500 0.026327522471547127
1800 0.010368862189352512
2100 0.038690391927957535
2400 0.012315887026488781
2700 0.0486142598092556
AB|AA
BCD|BCD
DAA|DAC


In [ ]:
print(predict("AB"))
print(predict("BCD"))
print(predict("DACBA"))

AB|AB
BCD|BCD
DAC|DAC


In [ ]:
with open("weights.txt", "w") as f:
    for name, param in model.named_parameters():
        f.write(f"{name}\n")
        f.write(f"shape = {tuple(param.shape)}\n")
        f.write(str(param.detach().numpy()))
        f.write("\n\n")